[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/10_tag_3_5_loss_regularisierung.ipynb)

# Tag 3-5 - 10 Loss-Funktionen und Regularisierung

Regularisierung soll nicht einfach irgendeinen Loss kleiner machen. Sie soll verhindern, dass ein zu flexibles Modell zufällige Details der Trainingsdaten auswendig lernt.

Dafür bauen wir hier ein bewusstes Overfitting-Szenario:

- Der Datensatz selbst ist lernbar.
- Nur im Trainingsset werden einige Labels absichtlich vertauscht.
- Ein großes Modell ohne Regularisierung kann diese falschen Labels teilweise auswendig lernen.
- L2-Regularisierung und Dropout sollen eine glattere Grenze lernen und dadurch besser generalisieren.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


## Wo wird der Loss eingestellt?

Beispiele:

- Binäre Klassifikation mit Sigmoid-Ausgabe: `loss=keras.losses.BinaryCrossentropy(from_logits=False)`
- Binäre Klassifikation ohne Sigmoid-Ausgabe: `loss=keras.losses.BinaryCrossentropy(from_logits=True)`
- Regression: `loss=keras.losses.MeanSquaredError()` oder `MeanAbsoluteError()`
- Multiklassenklassifikation: `CategoricalCrossentropy()` oder `SparseCategoricalCrossentropy()`


In [ ]:
X, y_clean = make_moons(n_samples=1400, noise=0.24, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X).astype("float32")
y_clean = y_clean.astype("float32")
y_underlying_all = y_clean.reshape(-1, 1)

X_train, X_rest, y_train_clean, y_rest = train_test_split(
    X, y_clean, train_size=120, random_state=RANDOM_STATE, stratify=y_clean
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest
)

rng = np.random.default_rng(RANDOM_STATE)
y_train_noisy = y_train_clean.copy()
noisy_idx = rng.choice(len(y_train_noisy), size=int(0.25 * len(y_train_noisy)), replace=False)
y_train_noisy[noisy_idx] = 1.0 - y_train_noisy[noisy_idx]

# Keras-BinaryCrossentropy erwartet hier denselben Rank wie der Output: (n, 1).
y_train_noisy = y_train_noisy.reshape(-1, 1)
y_train_clean = y_train_clean.reshape(-1, 1)
y_val = y_val.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train_noisy.ravel(), cmap="coolwarm", s=35, edgecolor="black", label="Train noisy")
plt.scatter(X_train[noisy_idx, 0], X_train[noisy_idx, 1], facecolors="none", edgecolors="yellow", s=120, linewidths=2, label="vertauschte Labels")
plt.title("Kleines Trainingsset mit absichtlich falschen Labels")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)
print("Anteil verrauschte Trainingslabels:", len(noisy_idx) / len(y_train_noisy))


## Was wird verglichen?

Alle Modelle nutzen dieselbe Binary-Cross-Entropy. Geändert wird nur die Regularisierung:

- **keine Regularisierung**: maximale Freiheit, hohes Overfitting-Risiko.
- **L2**: große Gewichte werden bestraft, die Grenze wird glatter.
- **L2 + Dropout**: zusätzlich werden im Training zufällig Aktivierungen deaktiviert.

Wichtig für die Interpretation: Das Modell trainiert auf verrauschten Labels, aber `val`, `test` und `underlying_clean` enthalten die sauberen Labels. Wenn Regularisierung funktioniert, muss sie nicht die beste Accuracy auf den falschen Trainingslabels erreichen. Sie soll näher an der zugrunde liegenden sauberen Funktion bleiben.


In [ ]:
def build_model(kind="keine"):
    reg = None
    dropout_rate = 0.0
    if kind == "L2":
        reg = regularizers.l2(3e-3)
    elif kind == "L2 + Dropout":
        reg = regularizers.l2(2e-3)
        dropout_rate = 0.35

    safe_kind = kind.lower().replace(" ", "_").replace("+", "plus")
    model = keras.Sequential(name=f"regularisierung_{safe_kind}")
    model.add(keras.Input(shape=(2,), name="features"))
    model.add(layers.Dense(128, activation="relu", kernel_regularizer=reg, name="hidden_1"))
    if dropout_rate:
        model.add(layers.Dropout(dropout_rate, name="dropout_1"))
    model.add(layers.Dense(128, activation="relu", kernel_regularizer=reg, name="hidden_2"))
    if dropout_rate:
        model.add(layers.Dropout(dropout_rate, name="dropout_2"))
    model.add(layers.Dense(64, activation="relu", kernel_regularizer=reg, name="hidden_3"))
    model.add(layers.Dense(1, activation="sigmoid", name="output"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=["accuracy", keras.metrics.BinaryCrossentropy(name="bce")],
    )
    return model


configs = ["keine", "L2", "L2 + Dropout"]
histories = {}
models = {}
summary = []

for name in configs:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(name)
    history = model.fit(
        X_train,
        y_train_noisy,
        validation_data=(X_val, y_val),
        epochs=180,
        batch_size=16,
        verbose=0,
    )
    train_loss, train_acc, train_bce = model.evaluate(X_train, y_train_noisy, verbose=0)
    clean_train_loss, clean_train_acc, clean_train_bce = model.evaluate(X_train, y_train_clean, verbose=0)
    test_loss, test_acc, test_bce = model.evaluate(X_test, y_test, verbose=0)
    underlying_loss, underlying_acc, underlying_bce = model.evaluate(X, y_underlying_all, verbose=0)
    histories[name] = history
    models[name] = model
    summary.append(
        {
            "modell": name,
            "train_acc_noisy": train_acc,
            "train_acc_clean": clean_train_acc,
            "best_val_acc": max(history.history["val_accuracy"]),
            "final_val_acc": history.history["val_accuracy"][-1],
            "test_acc": test_acc,
            "underlying_clean_acc": underlying_acc,
            "best_val_bce": min(history.history["val_bce"]),
            "underlying_clean_bce": underlying_bce,
        }
    )

summary_df = pd.DataFrame(summary).round(4)
display(summary_df)


## Trainingskurven und Vergleich zur sauberen Funktion

Wenn ein Modell falsche Trainingslabels auswendig lernt, kann Training Accuracy auf den verrauschten Labels steigen, während Validation Accuracy schlechter bleibt. Regularisierung soll diese Lücke reduzieren.

Zusätzlich vergleichen wir gegen `underlying_clean`: Das sind alle Datenpunkte mit den ursprünglichen, nicht vertauschten Labels. Das ist hier die beste Annäherung an die zugrunde liegende Funktion. Ein regularisiertes Modell kann auf noisy Training schlechter wirken, aber trotzdem näher an dieser sauberen Funktion liegen.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for name, history in histories.items():
    axes[0].plot(history.history["accuracy"], alpha=0.45, label=f"{name} train noisy")
    axes[0].plot(history.history["val_accuracy"], linewidth=2, label=f"{name} val")
    axes[1].plot(history.history["val_bce"], label=name)
    axes[2].plot(history.history["val_loss"], label=name)

axes[0].set_title("Accuracy: Train noisy vs. Validation")
axes[0].set_xlabel("Epoche")
axes[0].legend(fontsize=8)
axes[1].set_title("Validation BCE")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[2].set_title("Validation Loss inkl. Regularisierung")
axes[2].set_xlabel("Epoche")
axes[2].legend()

axes[3].bar(summary_df["modell"], summary_df["underlying_clean_acc"])
axes[3].set_ylim(0.0, 1.0)
axes[3].set_title("Accuracy gegen saubere Funktion")
axes[3].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## Entscheidungsgrenzen

Hier ist Regularisierung oft am verständlichsten: Ohne Regularisierung kann die Grenze kleine Beulen um einzelne Trainingspunkte bilden. Die gelb umrandeten Punkte sind absichtlich falsche Trainingslabels. Eine gute Regularisierung soll nicht jeden dieser Punkte auswendig lernen, sondern eine glattere Grenze finden, die zur sauberen Mondstruktur passt.

Das vierte Bild ist kein Modelloutput. Es zeigt den wahren Zusammenhang des Datensatzes mit den sauberen Labels. Genau daran sollen sich die regularisierten Modelle eher orientieren als an den absichtlich falschen Trainingslabels.


In [ ]:
def plot_boundary(model, title, ax):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 180),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 180),
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    proba = model.predict(grid, verbose=0).reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.35)
    ax.contour(xx, yy, proba, levels=[0.5], colors="black")
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train_noisy.ravel(), cmap="coolwarm", s=24, edgecolor="black")
    ax.scatter(X_train[noisy_idx, 0], X_train[noisy_idx, 1], facecolors="none", edgecolors="yellow", s=90, linewidths=1.8)
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


def plot_true_moons(ax):
    ax.scatter(X[:, 0], X[:, 1], c=y_underlying_all.ravel(), cmap="coolwarm", s=12, alpha=0.75)
    ax.set_title("Wahrer Zusammenhang\nclean Moons")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


fig, axes = plt.subplots(1, 4, figsize=(19, 4.5), sharex=True, sharey=True)
for ax, name in zip(axes[:3], configs):
    plot_boundary(models[name], name, ax)
plot_true_moons(axes[3])
plt.tight_layout()
plt.show()
